In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

file_path = "12_emt_scc_ID_score.h5ad"

adata = sc.read_h5ad(file_path)

print(adata)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '12_emt_scc_ID_score.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import matplotlib as plt

In [ ]:
print(adata.obsm.keys())

KeysView(AxisArrays with keys: 'X_harmony', 'X_pca', 'X_scanvi', 'X_scenic_pca', 'X_scenic_umap', 'X_umap', 'X_umap_scanvvi', 'X_unintegrated')


In [ ]:
sc.pl.umap(
    adata,
    color="ID_score",
    cmap="viridis",
    size=8,
    frameon=False,
    title="Intrinsic Dimension Score"
)

In [ ]:
order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    inner="box",
    cut=0
)

plt.xlabel("Cluster annotation")
plt.ylabel("Intrinsic Dimension Score")
plt.title("Intrinsic Dimension Score across Cluster Annotations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
print("X dtype:", adata.X.dtype)
print("X shape:", adata.X.shape)

if hasattr(adata.X, "min"):
    print("X min:", adata.X.min())
    print("X max:", adata.X.max())

print("\nLayers:", adata.layers.keys())
print("Raw available:", adata.raw is not None)

X dtype: float32
X shape: (23959, 20358)
X min: 0.0
X max: 19184.0

Layers: KeysView(Layers with keys: None)
Raw available: False


In [ ]:
import numpy as np
import scipy.sparse as sp

X = adata.X
n_cells = adata.n_obs

entropy = np.zeros(n_cells, dtype=np.float32)

chunk_size = 1000

for start in range(0, n_cells, chunk_size):
    end = min(start + chunk_size, n_cells)

    if sp.issparse(X):
        X_chunk = X[start:end].toarray()
    else:
        X_chunk = np.asarray(X[start:end])

    # Total transcripts per cell
    totals = X_chunk.sum(axis=1)

    # Calculate p = expression / total expression
    p = np.divide(
        X_chunk,
        totals[:, None],
        out=np.zeros_like(X_chunk, dtype=np.float32),
        where=totals[:, None] != 0
    )

    # Shannon entropy; p=0 contributes 0
    entropy[start:end] = -np.sum(
        np.where(p > 0, p * np.log(p), 0),
        axis=1
    )

    print(f"Processed {end:,} / {n_cells:,} cells")

adata.obs["transcriptome_entropy"] = entropy

/tmp/ipykernel_376805/3342716872.py:32: RuntimeWarning: divide by zero encountered in log
  np.where(p > 0, p * np.log(p), 0),
/tmp/ipykernel_376805/3342716872.py:32: RuntimeWarning: invalid value encountered in multiply
  np.where(p > 0, p * np.log(p), 0),


Processed 1,000 / 23,959 cells
Processed 2,000 / 23,959 cells
Processed 3,000 / 23,959 cells
Processed 4,000 / 23,959 cells
Processed 5,000 / 23,959 cells
Processed 6,000 / 23,959 cells
Processed 7,000 / 23,959 cells
Processed 8,000 / 23,959 cells
Processed 9,000 / 23,959 cells
Processed 10,000 / 23,959 cells
Processed 11,000 / 23,959 cells
Processed 12,000 / 23,959 cells
Processed 13,000 / 23,959 cells
Processed 14,000 / 23,959 cells
Processed 15,000 / 23,959 cells
Processed 16,000 / 23,959 cells
Processed 17,000 / 23,959 cells
Processed 18,000 / 23,959 cells
Processed 19,000 / 23,959 cells
Processed 20,000 / 23,959 cells
Processed 21,000 / 23,959 cells
Processed 22,000 / 23,959 cells
Processed 23,000 / 23,959 cells
Processed 23,959 / 23,959 cells


In [ ]:
print(adata.obs["transcriptome_entropy"].describe())

count    23959.000000
mean         7.198808
std          0.277241
min          4.590233
25%          7.011715
50%          7.202600
75%          7.390310
max          8.119326
Name: transcriptome_entropy, dtype: float64


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

plot_df = adata.obs[
    ["cluster_annotation", "transcriptome_entropy"]
].copy()

plot_df = plot_df.dropna()

# Keep only desired groups
plot_df = plot_df[
    plot_df["cluster_annotation"].isin(order)
].copy()

# ---------------------------------------------------------
# Overall Kruskal-Wallis test
# ---------------------------------------------------------

groups = [
    plot_df.loc[
        plot_df["cluster_annotation"] == group,
        "transcriptome_entropy"
    ].values
    for group in order
]

H, p_kw = kruskal(*groups)

print("Kruskal-Wallis test")
print("--------------------")
print(f"H = {H:.4f}")
print(f"p = {p_kw:.4e}")

# ---------------------------------------------------------
# Pairwise Mann-Whitney U tests
# ---------------------------------------------------------

pairwise_results = []

for i in range(len(order)):
    for j in range(i + 1, len(order)):

        group1 = plot_df.loc[
            plot_df["cluster_annotation"] == order[i],
            "transcriptome_entropy"
        ]

        group2 = plot_df.loc[
            plot_df["cluster_annotation"] == order[j],
            "transcriptome_entropy"
        ]

        U, p = mannwhitneyu(
            group1,
            group2,
            alternative="two-sided"
        )

        pairwise_results.append({
            "group1": order[i],
            "group2": order[j],
            "U": U,
            "p_raw": p
        })

pairwise_results = pd.DataFrame(pairwise_results)

# FDR correction
reject, p_adj, _, _ = multipletests(
    pairwise_results["p_raw"],
    method="fdr_bh"
)

pairwise_results["p_adj"] = p_adj
pairwise_results["significant"] = reject

print("\nPairwise comparisons")
print("--------------------")

display(pairwise_results)

Kruskal-Wallis test
--------------------
H = 796.3828
p = 0.0000e+00

Pairwise comparisons
--------------------


In [ ]:
plt.figure(figsize=(11, 6))

sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="transcriptome_entropy",
    order=order,
    inner="box",
    cut=0
)

plt.xlabel("Cluster annotation")
plt.ylabel("Transcriptome Shannon Entropy")
plt.title("Transcriptome Shannon Entropy across EMT States")

plt.text(
    0.98,
    0.98,
    f"Kruskal-Wallis: p = {p_kw:.2e}",
    transform=plt.gca().transAxes,
    ha="right",
    va="top",
    fontsize=11
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=plot_df,
    x="cluster_annotation",
    y="transcriptome_entropy",
    order=order,
    showfliers=False
)

plt.xlabel("Cluster annotation")
plt.ylabel("Transcriptome Shannon Entropy")
plt.title("Transcriptome Shannon Entropy across EMT States")

plt.text(
    0.98,
    0.98,
    f"Kruskal-Wallis: p = {p_kw:.2e}",
    transform=plt.gca().transAxes,
    ha="right",
    va="top",
    fontsize=11
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(
    adata.obs["ID_score"],
    bins=100,
    kde=True
)

plt.xlabel("Intrinsic Dimension Score")
plt.ylabel("Number of cells")
plt.title("Distribution of ID Score")
plt.show()

In [ ]:
adata.obs["ID_population"] = np.where(
    adata.obs["ID_score"] < 25,
    "Low-ID",
    "High-ID"
)

In [ ]:
sc.pl.umap(
    adata,
    color="ID_population",
    size=8,
    frameon=False,
    title="Intrinsic Dimension Populations"
)

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

# Get ID scores
id_scores = adata.obs["ID_score"].values.reshape(-1, 1)

# Fit 2-component Gaussian mixture
gmm = GaussianMixture(
    n_components=2,
    random_state=42
)

gmm.fit(id_scores)

# Population assignment
labels = gmm.predict(id_scores)

# Posterior probability of belonging to each population
probabilities = gmm.predict_proba(id_scores)

# Get the means of the two populations
means = gmm.means_.flatten()

print("GMM population means:")
print(means)

GMM population means:
[21.72385777 31.03267199]


In [ ]:
# Identify which GMM component is low and high
low_component = np.argmin(means)
high_component = np.argmax(means)

adata.obs["ID_population"] = np.where(
    labels == low_component,
    "Low-ID",
    "High-ID"
)

print(adata.obs["ID_population"].value_counts())

ID_population
Low-ID     14171
High-ID     9788
Name: count, dtype: int64


In [ ]:
sc.pl.umap(
    adata,
    color="ID_population",
    size=8,
    frameon=False,
    title="ID Score Populations"
)

In [ ]:
sc.pl.umap(
    adata,
    color="ID_score",
    cmap="viridis",
    size=8,
    frameon=False,
    title="Intrinsic Dimension Score"
)

In [ ]:
subset = adata[
    adata.obs["cluster_annotation"].isin(
        ["Epithelial", "Early-Hybrid"]
    )
].copy()

sc.pl.umap(
    subset,
    color="ID_population",
    size=10,
    frameon=False,
    title="ID Populations in Epithelial + Early-Hybrid"
)

In [ ]:
sc.pl.umap(
    subset,
    color="ID_score",
    cmap="viridis",
    size=10,
    frameon=False,
    title="ID Score in Epithelial + Early-Hybrid"
)

In [ ]:
from scipy import sparse

print("Sparse:", sparse.issparse(adata.X))
print("Non-zero entries:", adata.X.nnz if sparse.issparse(adata.X) else np.count_nonzero(adata.X))

Sparse: True
Non-zero entries: 102956885


In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
print(adata.layers["counts"].shape)
print(adata.layers["counts"].dtype)

(23959, 20358)
float32


In [ ]:
full_ccat2 = scent.run_scent_ccat(
    adata,
    species="mouse",
    counts_layer="counts",
    chunk_size=500
)

Using existing mapping: /home/csb/sampoorn/SCENT_pipeline/data/reference/mouse_human_1to1.csv
AnnData genes: 20358
Genes with strict 1:1 mapping: 13690
Genes passed toward SCENT: 13690
CCAT: cells 0 - 500 of 23,959
CCAT completed for 500 cells
CCAT: cells 500 - 1,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 1,000 - 1,500 of 23,959
CCAT completed for 500 cells
CCAT: cells 1,500 - 2,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 2,000 - 2,500 of 23,959
CCAT completed for 500 cells
CCAT: cells 2,500 - 3,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 3,000 - 3,500 of 23,959
CCAT completed for 500 cells
CCAT: cells 3,500 - 4,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 4,000 - 4,500 of 23,959
CCAT completed for 500 cells
CCAT: cells 4,500 - 5,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 5,000 - 5,500 of 23,959
CCAT completed for 500 cells
CCAT: cells 5,500 - 6,000 of 23,959
CCAT completed for 500 cells
CCAT: cells 6,000 - 6,500 of 23,959
CCAT co

In [ ]:
adata.write("12_emt_scc_ID_score.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

file_path = "12_emt_scc_ID_score.h5ad"

adata = sc.read_h5ad(file_path)

print(adata)

AnnData object with n_obs × n_vars = 23959 × 20358
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'phase', 'hypoxia', 'IFN_response', 'sample', 'LKPR_Epithelial_andrea', 'LKPR_EarlyHybridEMT_andrea', 'LKPR_HybridEMT_andrea', 'LKPR_LateEMT_andrea', 'clusterNr', 'cluster', 'clusterU', 'assay', 'type', 'nCount_RNA', 'nFeature_RNA', 'sample1', 'orig.bc', 'new.bcs', 'nCounts', 'nFeatures', 'perc_mt', 'perc_rp', 'clustering_res_0.2', 'clustering_res_0.3', 'clustering_res_0.4', 'clustering_res_0.5', 'clustering_res_0.6', 'clustering_res_0.7', 'clustering_res_0.8', 'clustering_res_0.9', 'seurat_clusters', 'clusterID', 'cluster_annotation', 'celltype_annotation', 'samplecluster', 'original_clustering', 'ID_score', 'transcriptome_entropy', 'ID_population', 'CCAT_score'
    var: 'mt', 'ribo', 'hb', 'highly_variable', 'means', 'dispersions', 'dispersions_n

In [ ]:
import matplotlib.pyplot as plt

In [ ]:


plot_df = adata.obs[["cluster_annotation", "CCAT_score"]].copy()

plot_df.head()

In [ ]:
plot_df1 = adata.obs[["cluster_annotation", "ID_score"]].copy()

plot_df1.head()

In [ ]:
import matplotlib.pyplot as plt

import seaborn as sns

import pandas as pd

order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=plot_df1,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    inner="box",
    cut=0
)

plt.xlabel("Cluster annotation")
plt.ylabel("ID Score")
plt.title("ID Score across Cluster Annotations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

import seaborn as sns

import pandas as pd

order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="CCAT_score",
    order=order,
    inner="box",
    cut=0
)

plt.xlabel("Cluster annotation")
plt.ylabel("SCENT Score")
plt.title("SCENT Score Score across Cluster Annotations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
print(adata.var_names[:20].tolist())

['0610009B22Rik', '0610009E02Rik', '0610009L18Rik', '0610009O20Rik', '0610010F05Rik', '0610010K14Rik', '0610012D04Rik', '0610012G03Rik', '0610030E20Rik', '0610031O16Rik', '0610033M10Rik', '0610037L13Rik', '0610038B21Rik', '0610039K10Rik', '0610040B10Rik', '0610040F04Rik', '0610040J01Rik', '0610043K17Rik', '1010001B22Rik', '1010001N08Rik']


In [ ]:
print(adata.var.columns.tolist())

['mt', 'ribo', 'hb', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std']


In [ ]:
ith1_human = [
    "CDKN1A", "LAMA3", "LAMC2", "PLAU", "COL17A1",
    "FAM83A", "INHBA", "LAMB3", "SEMA3C", "SERPINE1",
    "C15orf48", "CAB39", "CDH3", "GPR56", "IL32",
    "IRF6", "ITGA2", "ITGAV", "ITGB6", "JUP",
    "KLF6", "LRRC8A", "MAST4", "NDRG1", "NEAT1",
    "PIEZO1", "PTGS2", "RHOB", "SLFN5", "TGFA",
    "TJP2", "WNT7A", "ZBED2", "ACTR2", "AHNAK2",
    "AKAP12", "ANTXR2", "ATP2A2", "ATP6AP1", "BCAR3",
    "BRWD1", "BTG1", "C4orf3", "CAPG", "CCDC71L",
    "CDH1", "CLDN1", "CLDN4", "CLK1", "COPA"
]

print(len(ith1_human))

50


In [ ]:
ith1_mouse = [
    "Cdkn1a",
    "Lama3",
    "Lamc2",
    "Plau",
    "Col17a1",
    "Fam83a",
    "Inhba",
    "Lamb3",
    "Sema3c",
    "Serpine1",
    "Coxfa4l3",
    "Cab39",
    "Cdh3",
    "Adgrg1",
    "Irf6",
    "Itga2",
    "Itgav",
    "Itgb6",
    "Jup",
    "Klf6",
    "Lrrc8a",
    "Mast4",
    "Ndrg1",
    "Neat1",
    "Piezo1",
    "Ptgs2",
    "Rhob",
    "Slfn5",
    "Tgfa",
    "Tjp2",
    "Wnt7a",
    "Zbed2",
    "Actr2",
    "Ahnak2",
    "Akap12",
    "Antxr2",
    "Atp2a2",
    "Atp6ap1",
    "Bcar3",
    "Brwd1",
    "Btg1",
    "Arln",
    "Capg",
    "Ccdc71l",
    "Cdh1",
    "Cldn1",
    "Cldn4",
    "Clk1",
    "Copa"
]

In [ ]:
present_ith1 = [
    g for g in ith1_mouse
    if g in adata.var_names
]

missing_ith1 = [
    g for g in ith1_mouse
    if g not in adata.var_names
]

print("ITH1 mouse orthologs:", len(ith1_mouse))
print("Present in AnnData:", len(present_ith1))
print("Missing from AnnData:", len(missing_ith1))

print("\nPresent:")
print(present_ith1)

print("\nMissing:")
print(missing_ith1)

ITH1 mouse orthologs: 49
Present in AnnData: 46
Missing from AnnData: 3

Present:
['Cdkn1a', 'Lama3', 'Lamc2', 'Plau', 'Col17a1', 'Fam83a', 'Inhba', 'Lamb3', 'Sema3c', 'Serpine1', 'Cab39', 'Cdh3', 'Adgrg1', 'Irf6', 'Itga2', 'Itgav', 'Itgb6', 'Jup', 'Klf6', 'Lrrc8a', 'Mast4', 'Ndrg1', 'Neat1', 'Piezo1', 'Ptgs2', 'Rhob', 'Slfn5', 'Tgfa', 'Tjp2', 'Wnt7a', 'Actr2', 'Ahnak2', 'Akap12', 'Antxr2', 'Atp2a2', 'Atp6ap1', 'Bcar3', 'Brwd1', 'Btg1', 'Capg', 'Ccdc71l', 'Cdh1', 'Cldn1', 'Cldn4', 'Clk1', 'Copa']

Missing:
['Coxfa4l3', 'Zbed2', 'Arln']


In [ ]:
ith1_mouse_final = present_ith1

print("Final ITH1 genes used for ssGSEA:", len(ith1_mouse_final))
print(ith1_mouse_final)

Final ITH1 genes used for ssGSEA: 46
['Cdkn1a', 'Lama3', 'Lamc2', 'Plau', 'Col17a1', 'Fam83a', 'Inhba', 'Lamb3', 'Sema3c', 'Serpine1', 'Cab39', 'Cdh3', 'Adgrg1', 'Irf6', 'Itga2', 'Itgav', 'Itgb6', 'Jup', 'Klf6', 'Lrrc8a', 'Mast4', 'Ndrg1', 'Neat1', 'Piezo1', 'Ptgs2', 'Rhob', 'Slfn5', 'Tgfa', 'Tjp2', 'Wnt7a', 'Actr2', 'Ahnak2', 'Akap12', 'Antxr2', 'Atp2a2', 'Atp6ap1', 'Bcar3', 'Brwd1', 'Btg1', 'Capg', 'Ccdc71l', 'Cdh1', 'Cldn1', 'Cldn4', 'Clk1', 'Copa']


In [ ]:
ith1_gene_sets = {
    "ITH1": ith1_mouse_final
}

In [ ]:
test_adata = adata[:1000].copy()

print(test_adata.shape)

(1000, 20358)


In [ ]:
print(adata.obs.columns.tolist())

['orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'phase', 'hypoxia', 'IFN_response', 'sample', 'LKPR_Epithelial_andrea', 'LKPR_EarlyHybridEMT_andrea', 'LKPR_HybridEMT_andrea', 'LKPR_LateEMT_andrea', 'clusterNr', 'cluster', 'clusterU', 'assay', 'type', 'nCount_RNA', 'nFeature_RNA', 'sample1', 'orig.bc', 'new.bcs', 'nCounts', 'nFeatures', 'perc_mt', 'perc_rp', 'clustering_res_0.2', 'clustering_res_0.3', 'clustering_res_0.4', 'clustering_res_0.5', 'clustering_res_0.6', 'clustering_res_0.7', 'clustering_res_0.8', 'clustering_res_0.9', 'seurat_clusters', 'clusterID', 'cluster_annotation', 'celltype_annotation', 'samplecluster', 'original_clustering', 'ID_score', 'transcriptome_entropy', 'ID_population', 'CCAT_score']


In [ ]:
for col in [
    "clusterNr",
    "cluster",
    "clusterU",
    "clusterID",
    "cluster_annotation",
    "celltype_annotation",
    "ID_population"
]:
    print(f"\n--- {col} ---")
    print(adata.obs[col].value_counts(dropna=False).head(20))


--- clusterNr ---
clusterNr
0    4089
1    3635
2    3445
3    3378
4    3361
5    2291
6    2080
7    1145
8     535
Name: count, dtype: int64

--- cluster ---
cluster
Early Hybrid EMT    10175
Hybrid EMT           5715
Late EMT             4089
Epithelial           3980
Name: count, dtype: int64

--- clusterU ---
clusterU
Late EMT                          4089
Hybrid EMT                        3635
Epithelial                        3445
Proliferative Early Hybrid EMT    3378
Early Hybrid EMT_1                3361
Early Hybrid EMT_2                2291
Proliferative Hybrid EMT          2080
Early Hybrid EMT_3                1145
Epithelial (Differentiated)        535
Name: count, dtype: int64

--- clusterID ---
clusterID
c0    9985
c1    6220
c2    4146
c3    3608
Name: count, dtype: int64

--- cluster_annotation ---
cluster_annotation
Early-Hybrid    9985
Late-EMT        6220
Epithelial      4146
Hybrid          3608
Name: count, dtype: int64

--- celltype_annotation ---
celltype_an

In [ ]:
cluster_key = "cluster"

In [ ]:
ith1_mouse_final = present_ith1

In [ ]:
ith1_scores = np.full(adata.n_obs, np.nan, dtype=np.float32)

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp
from scipy import sparse

ith1_gene_sets = {
    "ITH1": present_ith1
}

batch_size = 1000

for start in range(0, adata.n_obs, batch_size):

    end = min(start + batch_size, adata.n_obs)

    print(f"Processing {start:,} - {end:,} / {adata.n_obs:,}")

    X_batch = adata.X[start:end]

    if sparse.issparse(X_batch):
        X_batch = X_batch.toarray()

    expr = pd.DataFrame(
        X_batch,
        index=adata.obs_names[start:end],
        columns=adata.var_names
    )

    # genes × cells
    expr = expr.T

    ss = gp.ssgsea(
        data=expr,
        gene_sets=ith1_gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        no_plot=True,
        threads=12
    )

    batch_scores = (
        ss.res2d
        .set_index("Name")["NES"]
        .reindex(adata.obs_names[start:end])
        .to_numpy(dtype=np.float32)
    )

    ith1_scores[start:end] = batch_scores

# Store in AnnData
adata.obs["ITH1_ssGSEA"] = ith1_scores

print("\nFinished!")
print(adata.obs["ITH1_ssGSEA"].describe())

Processing 0 - 1,000 / 23,959
Processing 1,000 - 2,000 / 23,959
Processing 2,000 - 3,000 / 23,959
Processing 3,000 - 4,000 / 23,959
Processing 4,000 - 5,000 / 23,959
Processing 5,000 - 6,000 / 23,959
Processing 6,000 - 7,000 / 23,959
Processing 7,000 - 8,000 / 23,959
Processing 8,000 - 9,000 / 23,959
Processing 9,000 - 10,000 / 23,959
Processing 10,000 - 11,000 / 23,959
Processing 11,000 - 12,000 / 23,959
Processing 12,000 - 13,000 / 23,959
Processing 13,000 - 14,000 / 23,959
Processing 14,000 - 15,000 / 23,959
Processing 15,000 - 16,000 / 23,959
Processing 16,000 - 17,000 / 23,959
Processing 17,000 - 18,000 / 23,959
Processing 18,000 - 19,000 / 23,959
Processing 19,000 - 20,000 / 23,959
Processing 20,000 - 21,000 / 23,959
Processing 21,000 - 22,000 / 23,959
Processing 22,000 - 23,000 / 23,959
Processing 23,000 - 23,959 / 23,959

Finished!
count    23959.000000
mean         0.422407
std          0.187160
min         -0.259805
25%          0.286886
50%          0.407974
75%          0.5

In [ ]:
adata.obs["ITH1_ssGSEA"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cluster_order = [
    "Epithelial",
    "Early Hybrid EMT",
    "Hybrid EMT",
    "Late EMT"
]

# Make sure the biological ordering is preserved
adata.obs["cluster"] = pd.Categorical(
    adata.obs["cluster"],
    categories=cluster_order,
    ordered=True
)

# Summary statistics
summary = (
    adata.obs
    .groupby("cluster", observed=True)["ITH1_ssGSEA"]
    .agg(["count", "mean", "median", "std"])
)

print(summary)

                  count      mean    median       std
cluster                                              
Epithelial         3980  0.534460  0.538772  0.189339
Early Hybrid EMT  10175  0.489858  0.500624  0.170984
Hybrid EMT         5715  0.315089  0.319472  0.136169
Late EMT           4089  0.295490  0.294408  0.126322


In [ ]:
plt.figure(figsize=(7, 5))

sns.violinplot(
    data=adata.obs,
    x="cluster",
    y="ITH1_ssGSEA",
    order=cluster_order,
    inner="box",
    cut=0
)

plt.xlabel("")
plt.ylabel("ITH1 ssGSEA NES")
plt.title("ITH1 program across EMT states")

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import scanpy as sc

sc.pl.umap(
    adata,
    color="ITH1_ssGSEA",
    cmap="viridis",
    size=12,
    frameon=False,
    title="ITH1 ssGSEA"
)

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

cluster_order = [
    "Epithelial",
    "Early Hybrid EMT",
    "Hybrid EMT",
    "Late EMT"
]

df = adata.obs[["cluster", "ITH1_ssGSEA"]].copy()
df = df.dropna()

# Make sure order is preserved
df["cluster"] = pd.Categorical(
    df["cluster"],
    categories=cluster_order,
    ordered=True
)

# All pairwise comparisons
comparisons = list(itertools.combinations(cluster_order, 2))

results = []

for group1, group2 in comparisons:

    x = df.loc[df["cluster"] == group1, "ITH1_ssGSEA"]
    y = df.loc[df["cluster"] == group2, "ITH1_ssGSEA"]

    stat, p = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    results.append({
        "group1": group1,
        "group2": group2,
        "U": stat,
        "pvalue": p
    })

results = pd.DataFrame(results)

# Benjamini-Hochberg correction
results["p_adj"] = multipletests(
    results["pvalue"],
    method="fdr_bh"
)[1]

print(results)

             group1            group2           U        pvalue         p_adj
0        Epithelial  Early Hybrid EMT  22873632.0  3.084824e-33  3.701789e-33
1        Epithelial        Hybrid EMT  18738232.0  0.000000e+00  0.000000e+00
2        Epithelial          Late EMT  13823560.0  0.000000e+00  0.000000e+00
3  Early Hybrid EMT        Hybrid EMT  45704960.0  0.000000e+00  0.000000e+00
4  Early Hybrid EMT          Late EMT  33951552.0  0.000000e+00  0.000000e+00
5        Hybrid EMT          Late EMT  12902020.0  1.225657e-18  1.225657e-18


In [ ]:
from statannotations.Annotator import Annotator
import matplotlib.pyplot as plt
import seaborn as sns

pairs = [
    ("Epithelial", "Early Hybrid EMT"),
    ("Epithelial", "Hybrid EMT"),
    ("Epithelial", "Late EMT"),
    ("Early Hybrid EMT", "Hybrid EMT"),
    ("Early Hybrid EMT", "Late EMT"),
    ("Hybrid EMT", "Late EMT")
]

plt.figure(figsize=(8, 6))

ax = sns.violinplot(
    data=df,
    x="cluster",
    y="ITH1_ssGSEA",
    order=cluster_order,
    inner="box",
    cut=0
)

annotator = Annotator(
    ax,
    pairs,
    data=df,
    x="cluster",
    y="ITH1_ssGSEA",
    order=cluster_order
)

annotator.configure(
    test="Mann-Whitney",
    text_format="star",
    comparisons_correction="BH",
    loc="outside",
    verbose=1
)

annotator.apply_and_annotate()

ax.set_xlabel("")
ax.set_ylabel("ITH1 ssGSEA NES")
ax.set_title("ITH1 program across EMT states")

sns.despine()
plt.tight_layout()
plt.show()

p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

Epithelial vs. Early Hybrid EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:3.085e-33 U_stat=2.287e+07
Early Hybrid EMT vs. Hybrid EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:0.000e+00 U_stat=4.570e+07
Hybrid EMT vs. Late EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:1.226e-18 U_stat=1.290e+07
Epithelial vs. Hybrid EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:0.000e+00 U_stat=1.874e+07
Early Hybrid EMT vs. Late EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:0.000e+00 U_stat=3.395e+07
Epithelial vs. Late EMT: Mann-Whitney-Wilcoxon test two-sided with Benjamini-Hochberg correction, P_val:0.000e+00 U_stat=1.382e+0

In [ ]:
import scanpy as sc

HPCS_genes = [
    "Slc4a11",
    "Il23a",
    "Rrad",
    "Tnf",
    "Abtb2",
    "Lif",
    "Tnfrsf11b",
    "Tigit",
    "Dusp10",
    "Ntm",
    "Trim15",
    "Tm4sf1",
    "Chac1",
    "Creb5",
    "5031415H12Rik",
    "Cxcl1",
    "Mcpt8",
    "Trp53inp1",
    "Abcg1",
    "Mapk11",
    "Klhl2",
    "Il24",
    "Zfand2a",
    "Tspan5",
    "Ddit3",
    "Ascc3",
    "Sgms2",
    "Grpel2",
    "Gdf15",
    "Dennd2c",
    "Ly6a",
    "Slc9a4",
    "Asns",
    "Ets1",
    "Ugcg",
    "Gcnt2",
    "Rnf19b",
    "Pvt1",
    "3110043O21Rik",
    "Cln8",
    "Hmces",
    "Maff",
    "Tnfaip3",
    "Prkx",
    "Tgfa",
    "Pxdc1",
    "Cdk17",
    "Slc1a4",
    "Arid5a",
    "Cldn4",
    "Iars",
    "Ptgs2",
    "Relb",
    "Mxd1",
    "Ppp1r13l",
    "Btg1",
    "Slc38a2",
    "Sema7a",
    "Smcr8",
    "Cwh43",
    "Aaed1",
    "B930095G15Rik",
    "Pvr",
    "Fbxo30",
    "Zdhhc5",
    "Chrnb1",
    "Rhbdf2",
    "Fam50a",
    "Cxcl16",
    "Itga2",
    "Hs3st1",
    "Trib1",
    "Tgif1",
    "B3gnt2",
    "Rdh10",
    "Rhob",
    "Gm28048",
    "Chmp4c",
    "Chic2",
    "Klf6",
    "Bcl10",
    "2610035D17Rik",
    "Gm10157",
    "Cars",
    "Bax",
    "Agap1",
    "Llph-ps2",
    "Otud4",
    "Prkca",
    "AI413582",
    "Procr",
    "9930111J21Rik1",
    "Fam83g",
    "Ocln",
    "1810013L24Rik",
    "Otud7b",
    "Actn1",
    "Gtpbp4",
    "Nars",
    "Pik3c2a",
    "Vps37b"
]

In [ ]:
present_genes_hpcs = [g for g in HPCS_genes if g in adata.var_names]
missing_genes = [g for g in HPCS_genes if g not in adata.var_names]

print(f"Total HPCS genes: {len(HPCS_genes)}")
print(f"Genes present: {len(present_genes)}")
print(f"Genes missing: {len(missing_genes)}")

print("\nMissing genes:")
print(missing_genes)

Total HPCS genes: 101


In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=present_genes_hpcs,
    score_name="HPCS_score",
    use_raw=False
)

In [ ]:
adata.obs["HPCS_score"].describe()

In [ ]:
adata.obs["cluster"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Desired biological order
cluster_order = [
    "Epithelial",
    "Early Hybrid EMT",
    "Hybrid EMT",
    "Late EMT"
]

plt.figure(figsize=(8, 5))

sns.violinplot(
    data=adata.obs,
    x="cluster",
    y="HPCS_score",
    order=cluster_order,
    inner="box",
    cut=0
)

plt.xlabel("")
plt.ylabel("HPCS score")
plt.title("HPCS program across EMT states")

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=ith1_mouse,
    score_name="ith1_score",
    use_raw=False
)

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp
from scipy import sparse

# Your HPCS gene set
hpcs_gene_sets = {
    "HPCS": present_genes_hpcs
}

print("HPCS genes used:", len(present_genes_hpcs))
print(present_genes_hpcs)

# Preallocate scores
hpcs_scores = np.full(
    adata.n_obs,
    np.nan,
    dtype=np.float32
)

batch_size = 1000

for start in range(0, adata.n_obs, batch_size):

    end = min(start + batch_size, adata.n_obs)

    print(
        f"Processing {start:,} - {end:,} / {adata.n_obs:,}"
    )

    # Expression matrix for this batch
    X_batch = adata.X[start:end]

    if sparse.issparse(X_batch):
        X_batch = X_batch.toarray()

    expr = pd.DataFrame(
        X_batch,
        index=adata.obs_names[start:end],
        columns=adata.var_names
    )

    # GSEApy expects genes × cells
    expr = expr.T

    # ssGSEA
    ss = gp.ssgsea(
        data=expr,
        gene_sets=hpcs_gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        no_plot=True,
        threads=32
    )

    # Extract normalized enrichment score
    batch_scores = (
        ss.res2d
        .set_index("Name")["NES"]
        .reindex(adata.obs_names[start:end])
        .to_numpy(dtype=np.float32)
    )

    hpcs_scores[start:end] = batch_scores


# Store in AnnData
adata.obs["HPCS_ssGSEA"] = hpcs_scores

print("\nFinished!")


HPCS genes used: 98
['Slc4a11', 'Il23a', 'Rrad', 'Tnf', 'Abtb2', 'Lif', 'Tnfrsf11b', 'Tigit', 'Dusp10', 'Ntm', 'Tm4sf1', 'Chac1', 'Creb5', '5031415H12Rik', 'Cxcl1', 'Mcpt8', 'Trp53inp1', 'Abcg1', 'Mapk11', 'Klhl2', 'Il24', 'Zfand2a', 'Tspan5', 'Ddit3', 'Ascc3', 'Sgms2', 'Grpel2', 'Gdf15', 'Dennd2c', 'Ly6a', 'Slc9a4', 'Asns', 'Ets1', 'Ugcg', 'Gcnt2', 'Rnf19b', 'Pvt1', '3110043O21Rik', 'Cln8', 'Hmces', 'Maff', 'Tnfaip3', 'Prkx', 'Tgfa', 'Pxdc1', 'Cdk17', 'Slc1a4', 'Arid5a', 'Cldn4', 'Iars', 'Ptgs2', 'Relb', 'Mxd1', 'Ppp1r13l', 'Btg1', 'Slc38a2', 'Sema7a', 'Smcr8', 'Cwh43', 'Aaed1', 'B930095G15Rik', 'Pvr', 'Fbxo30', 'Zdhhc5', 'Chrnb1', 'Rhbdf2', 'Fam50a', 'Cxcl16', 'Itga2', 'Hs3st1', 'Trib1', 'Tgif1', 'B3gnt2', 'Rdh10', 'Rhob', 'Gm28048', 'Chmp4c', 'Chic2', 'Klf6', 'Bcl10', '2610035D17Rik', 'Cars', 'Bax', 'Agap1', 'Otud4', 'Prkca', 'AI413582', 'Procr', '9930111J21Rik1', 'Fam83g', 'Ocln', '1810013L24Rik', 'Otud7b', 'Actn1', 'Gtpbp4', 'Nars', 'Pik3c2a', 'Vps37b']
Processing 0 - 1,000 / 23,9

In [ ]:
print(adata.obs["HPCS_ssGSEA"].describe())

count    23959.000000
mean         0.482992
std          0.168743
min         -0.161358
25%          0.365012
50%          0.476057
75%          0.596605
max          1.067182
Name: HPCS_ssGSEA, dtype: float64


In [ ]:
plt.figure(figsize=(8, 5))

sns.violinplot(
    data=adata.obs,
    x="cluster",
    y="HPCS_ssGSEA",
    order=cluster_order,
    inner="box",
    cut=0
)

plt.xlabel("")
plt.ylabel("HPCS ssGSEA score")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

cluster_order = [
    "Epithelial",
    "Early Hybrid EMT",
    "Hybrid EMT",
    "Late EMT"
]

# -------------------------------------------------------
# Prepare data
# -------------------------------------------------------

df = adata.obs[
    ["cluster", "HPCS_ssGSEA"]
].dropna().copy()

df["cluster"] = pd.Categorical(
    df["cluster"],
    categories=cluster_order,
    ordered=True
)

# -------------------------------------------------------
# Pairwise Mann–Whitney tests
# Descriptive within-tumor comparisons
# -------------------------------------------------------

results = []

for g1, g2 in combinations(cluster_order, 2):

    x = df.loc[
        df["cluster"] == g1,
        "HPCS_ssGSEA"
    ]

    y = df.loc[
        df["cluster"] == g2,
        "HPCS_ssGSEA"
    ]

    U, p = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    results.append({
        "group1": g1,
        "group2": g2,
        "n1": len(x),
        "n2": len(y),
        "U": U,
        "p": p
    })

results = pd.DataFrame(results)

# FDR correction across the six comparisons
results["FDR"] = multipletests(
    results["p"],
    method="fdr_bh"
)[1]

print(results)

             group1            group2     n1     n2           U             p  \
0        Epithelial  Early Hybrid EMT   3980  10175  17218546.0  1.093013e-43   
1        Epithelial        Hybrid EMT   3980   5715  11779726.0  2.688491e-03   
2        Epithelial          Late EMT   3980   4089   9845862.0  0.000000e+00   
3  Early Hybrid EMT        Hybrid EMT  10175   5715  34703672.0  0.000000e+00   
4  Early Hybrid EMT          Late EMT  10175   4089  28714864.0  0.000000e+00   
5        Hybrid EMT          Late EMT   5715   4089  13815866.0  0.000000e+00   

            FDR  
0  1.311615e-43  
1  2.688491e-03  
2  0.000000e+00  
3  0.000000e+00  
4  0.000000e+00  
5  0.000000e+00  


In [ ]:
# -------------------------------------------------------
# Publication-style visualization
# -------------------------------------------------------

sns.set_context("paper")
sns.set_style("white")

fig, ax = plt.subplots(figsize=(8, 5.5))

sns.violinplot(
    data=df,
    x="cluster",
    y="HPCS_ssGSEA",
    order=cluster_order,
    inner=None,
    cut=0,
    linewidth=1,
    ax=ax
)

# Median + IQR
sns.boxplot(
    data=df,
    x="cluster",
    y="HPCS_ssGSEA",
    order=cluster_order,
    width=0.18,
    showcaps=True,
    showfliers=False,
    boxprops={"facecolor": "white", "zorder": 3},
    whiskerprops={"linewidth": 1},
    medianprops={"color": "black", "linewidth": 1.5},
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("HPCS ssGSEA enrichment score")
ax.set_xticklabels(cluster_order, rotation=20)

sns.despine(ax=ax)

plt.tight_layout()
plt.show()

/tmp/ipykernel_785624/4259855687.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(cluster_order, rotation=20)


In [ ]:
score_col = "HPCS_ssGSEA"

summary = (
    adata.obs
    .groupby("cluster")[score_col]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std"
    )
)

print(summary)

                      n      mean    median       std
cluster                                              
Epithelial         3980  0.479565  0.469735  0.179966
Early Hybrid EMT  10175  0.522302  0.518694  0.161974
Hybrid EMT         5715  0.464328  0.457648  0.170550
Late EMT           4089  0.414597  0.410123  0.142745


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# EMT order shown in your figure
order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

# Keep cells with an EMT label and ID score
plot_df = (
    adata.obs[["cluster_annotation", "ID_score"]]
    .dropna()
    .query("cluster_annotation in @order")
    .copy()
)

# Number of cells in each cluster
count_df = (
    plot_df.groupby("cluster_annotation", observed=True)
    .size()
    .reindex(order, fill_value=0)
    .rename("n_cells")
    .reset_index()
)

display(count_df)

# Violin plot
fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    inner="box",
    cut=0,
    color="#2C78A8",
    ax=ax
)

# Add n to each x-axis label
labels = [
    f"{group}\n(n = {n:,})"
    for group, n in zip(count_df["cluster_annotation"], count_df["n_cells"])
]

ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_xlabel("Cluster annotation")
ax.set_ylabel("ID Score")
ax.set_title("ID Score across Cluster Annotations")

sns.despine()
plt.tight_layout()
plt.show()

/tmp/ipykernel_3753426/3601171079.py:47: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(labels, rotation=45, ha="right")


In [ ]:
# Median ID score per cluster
median_df = (
    plot_df.groupby("cluster_annotation", observed=True)["ID_score"]
    .median()
    .reindex(order)
    .reset_index(name="median_ID_score")
)

fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    inner="box",
    cut=0,
    color="#2C78A8",
    ax=ax
)

# Connect cluster medians
x_positions = range(len(order))

ax.plot(
    x_positions,
    median_df["median_ID_score"],
    color="crimson",
    marker="o",
    linewidth=2.5,
    markersize=7,
    label="Median ID score",
    zorder=10
)

# Add median values above the red points
for x, median in zip(x_positions, median_df["median_ID_score"]):
    ax.annotate(
        f"{median:.2f}",
        (x, median),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        color="crimson",
        fontsize=10,
        fontweight="bold"
    )

# Cluster names plus cell counts
labels = [
    f"{group}\n(n = {n:,})"
    for group, n in zip(count_df["cluster_annotation"], count_df["n_cells"])
]

ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_xlabel("Cluster annotation")
ax.set_ylabel("ID Score")
ax.set_title("ID Score across Cluster Annotations")
ax.legend(frameon=False)

sns.despine()
plt.tight_layout()
plt.show()

display(median_df)

/tmp/ipykernel_3753426/3394773105.py:55: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(labels, rotation=45, ha="right")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

plot_df = (
    adata.obs[["cluster_annotation", "ID_score"]]
    .dropna()
    .query("cluster_annotation in @order")
    .copy()
)

count_df = (
    plot_df.groupby("cluster_annotation", observed=True)
    .size()
    .reindex(order, fill_value=0)
    .rename("n_cells")
)

median_scores = (
    plot_df.groupby("cluster_annotation", observed=True)["ID_score"]
    .median()
    .reindex(order)
)

sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(9, 5.5))

# Light blue violins
sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    inner=None,
    cut=0,
    color="#4F83A5",
    linewidth=1.1,
    ax=ax
)

# Small white box plots inside each violin
sns.boxplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score",
    order=order,
    width=0.10,             # makes the box small
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "#222222",
        "linewidth": 1.1,
        "zorder": 4
    },
    whiskerprops={"color": "#222222", "linewidth": 1.1},
    capprops={"color": "#222222", "linewidth": 1.1},
    medianprops={"color": "#222222", "linewidth": 1.4},
    ax=ax
)

# Orange dashed line connecting cluster medians
x_positions = list(range(len(order)))
ax.plot(
    x_positions,
    median_scores.values,
    color="#ff7f0e",
    linestyle="--",
    marker="o",
    markersize=5,
    linewidth=1.5,
    zorder=6,
    label="Median ID score"
)

# Cluster labels with cell numbers
ax.set_xticklabels(
    [f"{group}\n(n = {count_df[group]:,})" for group in order],
    rotation=35,
    ha="right"
)

ax.set_xlabel("")
ax.set_ylabel("ID score")
ax.set_title("ID Score across EMT States", pad=10)
ax.legend(frameon=False, loc="upper right")

sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(count_df.rename_axis("Cluster").reset_index())
display(median_scores.rename("Median ID score").rename_axis("Cluster").reset_index())

/tmp/ipykernel_3753426/3698015807.py:79: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]

# Global min–max normalization: lowest cell = 0, highest cell = 1
id_min = adata.obs["ID_score"].min()
id_max = adata.obs["ID_score"].max()

adata.obs["ID_score_minmax"] = (
    (adata.obs["ID_score"] - id_min) / (id_max - id_min)
)

# Data used for the plot
plot_df = (
    adata.obs[["cluster_annotation", "ID_score_minmax"]]
    .dropna()
    .loc[lambda x: x["cluster_annotation"].isin(order)]
    .copy()
)

# Cells per EMT state
count_df = (
    plot_df.groupby("cluster_annotation", observed=True)
    .size()
    .reindex(order, fill_value=0)
)

# Median normalized ID score
median_scores = (
    plot_df.groupby("cluster_annotation", observed=True)["ID_score_minmax"]
    .median()
    .reindex(order)
)

sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(9, 5.5))

# Light-blue violin
sns.violinplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score_minmax",
    order=order,
    inner=None,
    cut=0,
    color="#4F83A5",
    linewidth=1.1,
    ax=ax
)

# Small white box plot
sns.boxplot(
    data=plot_df,
    x="cluster_annotation",
    y="ID_score_minmax",
    order=order,
    width=0.10,
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "#222222",
        "linewidth": 1.1,
        "zorder": 4
    },
    whiskerprops={"color": "#222222", "linewidth": 1.1},
    capprops={"color": "#222222", "linewidth": 1.1},
    medianprops={"color": "#222222", "linewidth": 1.4},
    ax=ax
)

# Orange dashed line joining medians
ax.plot(
    range(len(order)),
    median_scores.values,
    color="#ff7f0e",
    linestyle="--",
    marker="o",
    markersize=5,
    linewidth=1.5,
    zorder=6,
    label="Median"
)

# Cluster labels and cell counts
ax.set_xticklabels(
    [f"{group}\n(n = {count_df[group]:,})" for group in order],
    rotation=35,
    ha="right"
)

ax.set_xlabel("")
ax.set_ylabel("Min–max normalized ID score")
ax.set_title("Normalized ID Score across EMT States")
ax.set_ylim(-0.03, 1.08)
ax.legend(frameon=False, loc="upper right")

sns.despine(ax=ax)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3753426/2328264474.py:89: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(


In [ ]:
# If needed, run once:
# !pip install scikit-posthocs

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from scipy.stats import kruskal

# EMT-state order and columns
order = ["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"]
score_col = "ID_score_minmax"
group_col = "cluster_annotation"

# Ensure the plotting/test data are clean
plot_df = (
    adata.obs[[group_col, score_col]]
    .dropna()
    .loc[lambda x: x[group_col].isin(order)]
    .copy()
)

# Overall Kruskal–Wallis test
groups = [
    plot_df.loc[plot_df[group_col] == group, score_col]
    for group in order
]

H, p_kw = kruskal(*groups)
print(f"Kruskal–Wallis test: H = {H:.2f}, p = {p_kw:.3e}")

# Dunn's pairwise tests with Benjamini–Hochberg FDR correction
dunn_p = sp.posthoc_dunn(
    plot_df,
    val_col=score_col,
    group_col=group_col,
    p_adjust="fdr_bh"
)

# Put groups in biological order
dunn_p = dunn_p.reindex(index=order, columns=order)

# Show only the lower triangle
upper_triangle = np.triu(
    np.ones(dunn_p.shape, dtype=bool)
)
plot_p = dunn_p.mask(upper_triangle)

# Create readable p-value labels
labels = plot_p.copy().astype(object)

for row in plot_p.index:
    for col in plot_p.columns:
        p = plot_p.loc[row, col]

        if pd.isna(p):
            labels.loc[row, col] = ""
        elif p < 0.001:
            labels.loc[row, col] = f"{p:.1e}"
        else:
            labels.loc[row, col] = f"{p:.3f}"

# Plot pairwise-comparison heatmap
plt.figure(figsize=(7.5, 6))

sns.heatmap(
    plot_p.clip(upper=0.05),
    mask=plot_p.isna(),
    annot=labels,
    fmt="",
    cmap="viridis_r",
    vmin=0,
    vmax=0.05,
    square=True,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={"label": "Dunn test FDR-adjusted p-value"}
)

plt.title(
    "Normalized ID score: pairwise Dunn comparisons\n"
    f"Kruskal–Wallis p = {p_kw:.2e}"
)
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Full table of FDR-adjusted Dunn p-values
display(dunn_p)

Kruskal–Wallis test: H = 2590.78, p = 0.000e+00


In [ ]:
# If needed, run once:
# !pip install scikit-posthocs

import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from scipy.stats import kruskal

# EMT labels used in your gene-set analysis
group_col = "cluster"

order = [
    "Epithelial",
    "Early Hybrid EMT",
    "Hybrid EMT",
    "Late EMT"
]

def p_to_stars(p):
    """Convert FDR-adjusted p-values to significance labels."""
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"


def plot_geneset_by_emt(adata, score_col, y_label, title):
    # Prepare data
    df = (
        adata.obs[[group_col, score_col]]
        .dropna()
        .loc[lambda x: x[group_col].isin(order)]
        .copy()
    )

    # Counts and medians
    counts = (
        df.groupby(group_col, observed=True)
        .size()
        .reindex(order, fill_value=0)
    )

    medians = (
        df.groupby(group_col, observed=True)[score_col]
        .median()
        .reindex(order)
    )

    # Overall Kruskal–Wallis test
    groups = [
        df.loc[df[group_col] == group, score_col]
        for group in order
    ]

    H, p_kw = kruskal(*groups)

    # Dunn pairwise comparisons with FDR correction
    dunn_p = sp.posthoc_dunn(
        df,
        val_col=score_col,
        group_col=group_col,
        p_adjust="fdr_bh"
    ).reindex(index=order, columns=order)

    # Plot
    sns.set_theme(style="white", context="paper")
    fig, ax = plt.subplots(figsize=(9, 6.5))

    sns.violinplot(
        data=df,
        x=group_col,
        y=score_col,
        order=order,
        inner=None,
        cut=0,
        color="#4F83A5",
        linewidth=1.1,
        ax=ax
    )

    # Small white box plot
    sns.boxplot(
        data=df,
        x=group_col,
        y=score_col,
        order=order,
        width=0.10,
        showfliers=False,
        boxprops={
            "facecolor": "white",
            "edgecolor": "#222222",
            "linewidth": 1.1,
            "zorder": 4
        },
        whiskerprops={"color": "#222222", "linewidth": 1.1},
        capprops={"color": "#222222", "linewidth": 1.1},
        medianprops={"color": "#222222", "linewidth": 1.4},
        ax=ax
    )

    # Median trend
    ax.plot(
        range(len(order)),
        medians.values,
        color="#ff7f0e",
        linestyle="--",
        marker="o",
        markersize=5,
        linewidth=1.5,
        zorder=6,
        label="Median"
    )

    # Add Dunn-test brackets for significant comparisons
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    bracket_step = y_range * 0.11
    bracket_height = y_range * 0.018
    bracket_number = 0

    for group1, group2 in itertools.combinations(order, 2):
        p_adj = dunn_p.loc[group1, group2]

        # Show only FDR-significant comparisons
        if p_adj < 0.05:
            x1 = order.index(group1)
            x2 = order.index(group2)
            y = y_max + bracket_step * (bracket_number + 0.4)

            ax.plot(
                [x1, x1, x2, x2],
                [y, y + bracket_height, y + bracket_height, y],
                color="#222222",
                linewidth=1
            )

            ax.text(
                (x1 + x2) / 2,
                y + bracket_height,
                p_to_stars(p_adj),
                ha="center",
                va="bottom",
                fontsize=10
            )

            bracket_number += 1

    # Axis labels
    ax.set_xticklabels(
        [f"{group}\n(n = {counts[group]:,})" for group in order],
        rotation=30,
        ha="right"
    )

    ax.set_xlabel("")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.text(
        0.02, 0.98,
        f"Kruskal–Wallis: H = {H:.2f}, p = {p_kw:.2e}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9
    )

    # Make room for significance brackets
    if bracket_number > 0:
        ax.set_ylim(
            y_min,
            y_max + bracket_step * (bracket_number + 1.2)
        )

    ax.legend(frameon=False, loc="lower right")
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

    # Return the FDR-adjusted comparison table
    return dunn_p

In [ ]:
import numpy as np
import pandas as pd
import gseapy as gp
from scipy import sparse

# ----------------------------------------------------------
# MP14 EMT-III: human genes converted to mouse orthologs
# IL32 and S100P are intentionally excluded (no reliable
# one-to-one mouse ortholog).
# ----------------------------------------------------------
mp14_emt3_mouse = [
    "Anxa2",
    "Krt19",
    "S100a6",
    "Tnfrsf12a",
    "Krt7",
    "Lgals1",
    "Anxa1",
    "Tm4sf1",
    "S100a10",
    "Lgals3",
    "S100a16",
    "Tagln2",
    "Lmna",
    "Crip1",
    "Ezr",
    "Krt18",
    "S100a11",
    "S100a4",
    "Emp3",
    "Hspb1",
    "Krt8",
    "Tuba1a",
    "Vim",
    "Klf6",
    "Anxa3",
    "Sh3bgrl3",
    "Tmsb4x",
    "2200002D01Rik",  # human C19ORF33
    "Ckb",
    "Clic1",
    "Emp1",
    "Ifi27",
    "Isg15",
    "Mdk",
    "Mmp7",
    "Phlda2",
    "Plp2",
    "Cldn4",
    "Myadm",
    "Rhoc",
    "S100a14",
    "Cd24a",           # human CD24
    "Crip2",
    "Myl12a",
    "Rab11fip1",
    "Stmn1",
    "Fxyd3",
    "Flna"
]

# Keep only genes that exist in your dataset
mp14_present = [
    gene for gene in mp14_emt3_mouse
    if gene in adata.var_names
]

mp14_missing = [
    gene for gene in mp14_emt3_mouse
    if gene not in adata.var_names
]

print(f"MP14 mouse genes supplied: {len(mp14_emt3_mouse)}")
print(f"MP14 genes present in adata: {len(mp14_present)}")

print("\nGenes used for MP14 ssGSEA:")
print(mp14_present)

print("\nGenes absent from this dataset:")
print(mp14_missing)

# Stop if too few genes were found
if len(mp14_present) < 10:
    raise ValueError(
        "Fewer than 10 MP14 genes were found. Check whether adata.var_names "
        "contains mouse gene symbols rather than Ensembl IDs."
    )

# ----------------------------------------------------------
# Run MP14 ssGSEA cell by cell, in batches
# ----------------------------------------------------------
mp14_gene_sets = {
    "MP14_EMTIII": mp14_present
}

mp14_scores = np.full(
    adata.n_obs,
    np.nan,
    dtype=np.float32
)

batch_size = 1000

for start in range(0, adata.n_obs, batch_size):
    end = min(start + batch_size, adata.n_obs)

    print(f"Processing {start:,}–{end:,} of {adata.n_obs:,} cells")

    X_batch = adata.X[start:end]

    if sparse.issparse(X_batch):
        X_batch = X_batch.toarray()

    # ssGSEA requires genes as rows and cells as columns
    expression_df = pd.DataFrame(
        X_batch,
        index=adata.obs_names[start:end],
        columns=adata.var_names
    ).T

    ss = gp.ssgsea(
        data=expression_df,
        gene_sets=mp14_gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        no_plot=True,
        outdir=None,
        threads=36
    )

    batch_scores = (
        ss.res2d
        .set_index("Name")["NES"]
        .reindex(adata.obs_names[start:end])
        .to_numpy(dtype=np.float32)
    )

    mp14_scores[start:end] = batch_scores

# Save MP14 score in AnnData
adata.obs["MP14_EMTIII_ssGSEA"] = mp14_scores

print("\nMP14 EMT-III ssGSEA score summary:")
display(adata.obs["MP14_EMTIII_ssGSEA"].describe())

# Save a new file so this score is retained
adata.write("12_emt_scc_ID_score_enrichment_scores.h5ad")
print("Saved MP14 score successfully.")

MP14 mouse genes supplied: 48
MP14 genes present in adata: 47

Genes used for MP14 ssGSEA:
['Anxa2', 'Krt19', 'S100a6', 'Tnfrsf12a', 'Krt7', 'Lgals1', 'Anxa1', 'Tm4sf1', 'S100a10', 'Lgals3', 'S100a16', 'Tagln2', 'Lmna', 'Crip1', 'Ezr', 'Krt18', 'S100a11', 'S100a4', 'Emp3', 'Hspb1', 'Krt8', 'Tuba1a', 'Vim', 'Klf6', 'Anxa3', 'Sh3bgrl3', 'Tmsb4x', '2200002D01Rik', 'Ckb', 'Clic1', 'Emp1', 'Ifi27', 'Isg15', 'Mdk', 'Phlda2', 'Plp2', 'Cldn4', 'Myadm', 'Rhoc', 'S100a14', 'Cd24a', 'Crip2', 'Myl12a', 'Rab11fip1', 'Stmn1', 'Fxyd3', 'Flna']

Genes absent from this dataset:
['Mmp7']
Processing 0–1,000 of 23,959 cells
Processing 1,000–2,000 of 23,959 cells
Processing 2,000–3,000 of 23,959 cells
Processing 3,000–4,000 of 23,959 cells
Processing 4,000–5,000 of 23,959 cells
Processing 5,000–6,000 of 23,959 cells
Processing 6,000–7,000 of 23,959 cells
Processing 7,000–8,000 of 23,959 cells
Processing 8,000–9,000 of 23,959 cells
Processing 9,000–10,000 of 23,959 cells
Processing 10,000–11,000 of 23,959 c

Saved MP14 score successfully.


In [ ]:
mp14_dunn_results = plot_geneset_by_emt(
    adata,
    score_col="MP14_EMTIII_ssGSEA",
    y_label="MP14 EMT-III ssGSEA enrichment score",
    title="MP14 EMT-III program across EMT states"
)

display(mp14_dunn_results)

/tmp/ipykernel_3753426/4202973876.py:157: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(


In [ ]:
%pip install -q gseapy scikit-posthocs

/bin/bash: /home/csb/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp

from scipy import sparse
from scipy.stats import kruskal
from itertools import combinations

# Change this only if your .h5ad file is in another folder
file_path = "12_emt_scc_ID_score.h5ad"

adata = sc.read_h5ad(file_path)

print(adata)
print("\nAvailable metadata columns:")
print(adata.obs.columns.tolist())

AnnData object with n_obs × n_vars = 23959 × 20358
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'phase', 'hypoxia', 'IFN_response', 'sample', 'LKPR_Epithelial_andrea', 'LKPR_EarlyHybridEMT_andrea', 'LKPR_HybridEMT_andrea', 'LKPR_LateEMT_andrea', 'clusterNr', 'cluster', 'clusterU', 'assay', 'type', 'nCount_RNA', 'nFeature_RNA', 'sample1', 'orig.bc', 'new.bcs', 'nCounts', 'nFeatures', 'perc_mt', 'perc_rp', 'clustering_res_0.2', 'clustering_res_0.3', 'clustering_res_0.4', 'clustering_res_0.5', 'clustering_res_0.6', 'clustering_res_0.7', 'clustering_res_0.8', 'clustering_res_0.9', 'seurat_clusters', 'clusterID', 'cluster_annotation', 'celltype_annotation', 'samplecluster', 'original_clustering', 'ID_score', 'transcriptome_entropy', 'ID_population', 'CCAT_score'
    var: 'mt', 'ribo', 'hb', 'highly_variable', 'means', 'dispersions', 'dispersions_n

In [ ]:
HPCS_genes = [
    "Slc4a11", "Il23a", "Rrad", "Tnf", "Abtb2", "Lif",
    "Tnfrsf11b", "Tigit", "Dusp10", "Ntm", "Trim15",
    "Tm4sf1", "Chac1", "Creb5", "5031415H12Rik", "Cxcl1",
    "Mcpt8", "Trp53inp1", "Abcg1", "Mapk11", "Klhl2",
    "Il24", "Zfand2a", "Tspan5", "Ddit3", "Ascc3",
    "Sgms2", "Grpel2", "Gdf15", "Dennd2c", "Ly6a",
    "Slc9a4", "Asns", "Ets1", "Ugcg", "Gcnt2", "Rnf19b",
    "Pvt1", "3110043O21Rik", "Cln8", "Hmces", "Maff",
    "Tnfaip3", "Prkx", "Tgfa", "Pxdc1", "Cdk17",
    "Slc1a4", "Arid5a", "Cldn4", "Iars", "Ptgs2",
    "Relb", "Mxd1", "Ppp1r13l", "Btg1", "Slc38a2",
    "Sema7a", "Smcr8", "Cwh43", "Aaed1", "B930095G15Rik",
    "Pvr", "Fbxo30", "Zdhhc5", "Chrnb1", "Rhbdf2",
    "Fam50a", "Cxcl16", "Itga2", "Hs3st1", "Trib1",
    "Tgif1", "B3gnt2", "Rhob", "Gm28048", "Chmp4c",
    "Chic2", "Klf6", "Bcl10", "2610035D17Rik", "Gm10157",
    "Cars", "Bax", "Agap1", "Llph-ps2", "Otud4",
    "Prkca", "AI413582", "Procr", "9930111J21Rik1",
    "Fam83g", "Ocln", "1810013L24Rik", "Otud7b",
    "Actn1", "Gtpbp4", "Nars", "Pik3c2a", "Vps37b"
]

hpcs_present = [gene for gene in HPCS_genes if gene in adata.var_names]
hpcs_missing = [gene for gene in HPCS_genes if gene not in adata.var_names]

print(f"HPCS genes supplied: {len(HPCS_genes)}")
print(f"HPCS genes present in your data: {len(hpcs_present)}")
print(f"HPCS genes missing from your data: {len(hpcs_missing)}")

print("\nGenes used for scoring:")
print(hpcs_present)

HPCS genes supplied: 100
HPCS genes present in your data: 97
HPCS genes missing from your data: 3

Genes used for scoring:
['Slc4a11', 'Il23a', 'Rrad', 'Tnf', 'Abtb2', 'Lif', 'Tnfrsf11b', 'Tigit', 'Dusp10', 'Ntm', 'Tm4sf1', 'Chac1', 'Creb5', '5031415H12Rik', 'Cxcl1', 'Mcpt8', 'Trp53inp1', 'Abcg1', 'Mapk11', 'Klhl2', 'Il24', 'Zfand2a', 'Tspan5', 'Ddit3', 'Ascc3', 'Sgms2', 'Grpel2', 'Gdf15', 'Dennd2c', 'Ly6a', 'Slc9a4', 'Asns', 'Ets1', 'Ugcg', 'Gcnt2', 'Rnf19b', 'Pvt1', '3110043O21Rik', 'Cln8', 'Hmces', 'Maff', 'Tnfaip3', 'Prkx', 'Tgfa', 'Pxdc1', 'Cdk17', 'Slc1a4', 'Arid5a', 'Cldn4', 'Iars', 'Ptgs2', 'Relb', 'Mxd1', 'Ppp1r13l', 'Btg1', 'Slc38a2', 'Sema7a', 'Smcr8', 'Cwh43', 'Aaed1', 'B930095G15Rik', 'Pvr', 'Fbxo30', 'Zdhhc5', 'Chrnb1', 'Rhbdf2', 'Fam50a', 'Cxcl16', 'Itga2', 'Hs3st1', 'Trib1', 'Tgif1', 'B3gnt2', 'Rhob', 'Gm28048', 'Chmp4c', 'Chic2', 'Klf6', 'Bcl10', '2610035D17Rik', 'Cars', 'Bax', 'Agap1', 'Otud4', 'Prkca', 'AI413582', 'Procr', '9930111J21Rik1', 'Fam83g', 'Ocln', '1810013

In [ ]:
if len(hpcs_present) < 10:
    raise ValueError(
        "Too few HPCS genes were found. Check that adata.var_names "
        "contains mouse gene symbols."
    )

hpcs_gene_sets = {
    "HPCS": hpcs_present
}

hpcs_scores = np.full(
    adata.n_obs,
    np.nan,
    dtype=np.float32
)

batch_size = 1000

for start in range(0, adata.n_obs, batch_size):
    end = min(start + batch_size, adata.n_obs)

    print(f"Processing cells {start:,}–{end:,} of {adata.n_obs:,}")

    X_batch = adata.X[start:end]

    if sparse.issparse(X_batch):
        X_batch = X_batch.toarray()

    # ssGSEA requires: rows = genes, columns = cells
    expression_df = pd.DataFrame(
        X_batch,
        index=adata.obs_names[start:end],
        columns=adata.var_names
    ).T

    ss = gp.ssgsea(
        data=expression_df,
        gene_sets=hpcs_gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        no_plot=True,
        outdir=None,
        threads=4
    )

    batch_scores = (
        ss.res2d
        .set_index("Name")["NES"]
        .reindex(adata.obs_names[start:end])
        .to_numpy(dtype=np.float32)
    )

    hpcs_scores[start:end] = batch_scores

adata.obs["HPCS_ssGSEA"] = hpcs_scores

print("\nHPCS ssGSEA score summary:")
display(adata.obs["HPCS_ssGSEA"].describe())

# Save to a new file so your original file is preserved
output_file = "12_emt_scc_ID_score_HPCS_ssGSEA.h5ad"
adata.write(output_file)

print(f"\nSaved scored data as: {output_file}")

Processing cells 0–1,000 of 23,959
Processing cells 1,000–2,000 of 23,959
Processing cells 2,000–3,000 of 23,959
Processing cells 3,000–4,000 of 23,959
Processing cells 4,000–5,000 of 23,959
Processing cells 5,000–6,000 of 23,959
Processing cells 6,000–7,000 of 23,959
Processing cells 7,000–8,000 of 23,959
Processing cells 8,000–9,000 of 23,959
Processing cells 9,000–10,000 of 23,959
Processing cells 10,000–11,000 of 23,959
Processing cells 11,000–12,000 of 23,959
Processing cells 12,000–13,000 of 23,959
Processing cells 13,000–14,000 of 23,959
Processing cells 14,000–15,000 of 23,959
Processing cells 15,000–16,000 of 23,959
Processing cells 16,000–17,000 of 23,959
Processing cells 17,000–18,000 of 23,959
Processing cells 18,000–19,000 of 23,959
Processing cells 19,000–20,000 of 23,959
Processing cells 20,000–21,000 of 23,959
Processing cells 21,000–22,000 of 23,959
Processing cells 22,000–23,000 of 23,959
Processing cells 23,000–23,959 of 23,959

HPCS ssGSEA score summary:



Saved scored data as: 12_emt_scc_ID_score_HPCS_ssGSEA.h5ad


In [ ]:
# Automatically identify which EMT annotation column is usable
if (
    "cluster" in adata.obs.columns
    and set(["Epithelial", "Early Hybrid EMT", "Hybrid EMT", "Late EMT"])
    .issubset(set(adata.obs["cluster"].dropna()))
):
    group_col = "cluster"
    order = [
        "Epithelial",
        "Early Hybrid EMT",
        "Hybrid EMT",
        "Late EMT"
    ]

elif (
    "cluster_annotation" in adata.obs.columns
    and set(["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"])
    .issubset(set(adata.obs["cluster_annotation"].dropna()))
):
    group_col = "cluster_annotation"
    order = [
        "Epithelial",
        "Early-Hybrid",
        "Hybrid",
        "Late-EMT"
    ]

else:
    raise ValueError(
        "Could not find the expected EMT labels in either "
        "'cluster' or 'cluster_annotation'."
    )

score_col = "HPCS_ssGSEA"

plot_df = (
    adata.obs[[group_col, score_col]]
    .dropna()
    .loc[lambda x: x[group_col].isin(order)]
    .copy()
)

# Number of cells and median score per EMT state
counts = (
    plot_df.groupby(group_col, observed=True)
    .size()
    .reindex(order, fill_value=0)
)

medians = (
    plot_df.groupby(group_col, observed=True)[score_col]
    .median()
    .reindex(order)
)

# Overall Kruskal–Wallis test
groups = [
    plot_df.loc[plot_df[group_col] == group, score_col]
    for group in order
]

H, p_kw = kruskal(*groups)

# Pairwise Dunn tests with FDR correction
dunn_p = sp.posthoc_dunn(
    plot_df,
    val_col=score_col,
    group_col=group_col,
    p_adjust="fdr_bh"
).reindex(index=order, columns=order)

display(dunn_p)

def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"

# Plot
sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(9, 6.5))

sns.violinplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    inner=None,
    cut=0,
    color="#4F83A5",
    linewidth=1.1,
    ax=ax
)

sns.boxplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    width=0.10,
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "#222222",
        "linewidth": 1.1,
        "zorder": 4
    },
    whiskerprops={"color": "#222222", "linewidth": 1.1},
    capprops={"color": "#222222", "linewidth": 1.1},
    medianprops={"color": "#222222", "linewidth": 1.4},
    ax=ax
)

# Orange dashed line joining the medians
ax.plot(
    range(len(order)),
    medians.values,
    color="#ff7f0e",
    linestyle="--",
    marker="o",
    markersize=5,
    linewidth=1.5,
    zorder=6,
    label="Median"
)

# Add FDR-significant Dunn-test brackets
y_min, y_max = ax.get_ylim()
y_range = y_max - y_min
step = y_range * 0.11
height = y_range * 0.018
bracket_number = 0

for group1, group2 in combinations(order, 2):
    p_adj = dunn_p.loc[group1, group2]

    if p_adj < 0.05:
        x1 = order.index(group1)
        x2 = order.index(group2)
        y = y_max + step * (bracket_number + 0.4)

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + height, y + height, y],
            color="#222222",
            linewidth=1
        )

        ax.text(
            (x1 + x2) / 2,
            y + height,
            p_to_stars(p_adj),
            ha="center",
            va="bottom",
            fontsize=10
        )

        bracket_number += 1

# Labels
ax.set_xticks(
    range(len(order)),
    [f"{group}\n(n = {counts[group]:,})" for group in order],
    rotation=30,
    ha="right"
)

ax.set_xlabel("")
ax.set_ylabel("HPCS ssGSEA enrichment score")
ax.set_title("HPCS Program across EMT States")

ax.text(
    0.02,
    0.98,
    f"Kruskal–Wallis: H = {H:.2f}, p = {p_kw:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9
)

if bracket_number > 0:
    ax.set_ylim(y_min, y_max + step * (bracket_number + 1.2))

ax.legend(frameon=False, loc="lower right")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# If this import fails, run once:
# %pip install -q scikit-posthocs

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp

from scipy.stats import kruskal

# ----------------------------------------------------------
# Choose the EMT annotation column automatically
# ----------------------------------------------------------
if (
    "cluster" in adata.obs.columns
    and set(["Epithelial", "Early Hybrid EMT", "Hybrid EMT", "Late EMT"])
    .issubset(set(adata.obs["cluster"].dropna()))
):
    group_col = "cluster"
    order = [
        "Epithelial",
        "Early Hybrid EMT",
        "Hybrid EMT",
        "Late EMT"
    ]

elif (
    "cluster_annotation" in adata.obs.columns
    and set(["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"])
    .issubset(set(adata.obs["cluster_annotation"].dropna()))
):
    group_col = "cluster_annotation"
    order = [
        "Epithelial",
        "Early-Hybrid",
        "Hybrid",
        "Late-EMT"
    ]

else:
    raise ValueError("Could not find the expected EMT-state labels.")

score_col = "HPCS_ssGSEA"

if score_col not in adata.obs.columns:
    raise ValueError(
        "HPCS_ssGSEA is not in adata.obs. Run the HPCS scoring cell first."
    )

# ----------------------------------------------------------
# Prepare data, medians, and cell counts
# ----------------------------------------------------------
plot_df = (
    adata.obs[[group_col, score_col]]
    .dropna()
    .loc[lambda x: x[group_col].isin(order)]
    .copy()
)

counts = (
    plot_df.groupby(group_col, observed=True)
    .size()
    .reindex(order, fill_value=0)
)

medians = (
    plot_df.groupby(group_col, observed=True)[score_col]
    .median()
    .reindex(order)
)

summary = (
    plot_df.groupby(group_col, observed=True)[score_col]
    .agg(n="count", median="median", mean="mean", std="std")
    .reindex(order)
)

display(summary)

# ----------------------------------------------------------
# Kruskal–Wallis and Dunn post-hoc comparisons
# ----------------------------------------------------------
groups = [
    plot_df.loc[plot_df[group_col] == group, score_col]
    for group in order
]

H, p_kw = kruskal(*groups)

dunn_p = sp.posthoc_dunn(
    plot_df,
    val_col=score_col,
    group_col=group_col,
    p_adjust="fdr_bh"
).reindex(index=order, columns=order)

print(f"Kruskal–Wallis: H = {H:.2f}")
print("Kruskal–Wallis p-value:", "< 1e-300" if p_kw == 0 else f"{p_kw:.3e}")

display(dunn_p)

def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"

# ----------------------------------------------------------
# Publication-style HPCS plot
# ----------------------------------------------------------
sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(8.5, 5.8))

# Light-blue violin
sns.violinplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    inner=None,
    cut=0,
    color="#5B89A6",
    linewidth=1.1,
    ax=ax
)

# Small white boxplot
sns.boxplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    width=0.10,
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "#222222",
        "linewidth": 1.1,
        "zorder": 4
    },
    whiskerprops={"color": "#222222", "linewidth": 1.1},
    capprops={"color": "#222222", "linewidth": 1.1},
    medianprops={"color": "#222222", "linewidth": 1.4},
    ax=ax
)

# Orange line joining median HPCS scores
ax.plot(
    range(len(order)),
    medians.values,
    color="#ff7f0e",
    linestyle="--",
    marker="o",
    markersize=5.5,
    linewidth=1.6,
    zorder=6,
    label="Median"
)

# ----------------------------------------------------------
# Show ONLY Early Hybrid EMT comparisons
# ----------------------------------------------------------
early_hybrid = order[1]

comparisons_to_show = [
    (order[0], early_hybrid),
    (early_hybrid, order[2]),
    (early_hybrid, order[3])
]

y_min, y_max = ax.get_ylim()
y_range = y_max - y_min
step = y_range * 0.075
height = y_range * 0.016

for number, (group1, group2) in enumerate(comparisons_to_show):
    p_adj = dunn_p.loc[group1, group2]

    x1 = order.index(group1)
    x2 = order.index(group2)
    y = y_max + step * (number + 0.4)

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + height, y + height, y],
        color="#222222",
        linewidth=1
    )

    ax.text(
        (x1 + x2) / 2,
        y + height,
        p_to_stars(p_adj),
        ha="center",
        va="bottom",
        fontsize=10
    )

# Labels and final styling
ax.set_xticks(
    range(len(order)),
    [f"{group}\n(n = {counts[group]:,})" for group in order],
    rotation=30,
    ha="right"
)

ax.set_xlabel("")
ax.set_ylabel("HPCS ssGSEA enrichment score")
ax.set_title("HPCS Program across EMT States", pad=10)

kw_text = "p < 1e-300" if p_kw == 0 else f"p = {p_kw:.2e}"

ax.text(
    0.02,
    0.98,
    f"Kruskal–Wallis: H = {H:.2f}, {kw_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9
)

ax.set_ylim(y_min, y_max + step * 4.3)
ax.legend(frameon=False, loc="lower right")

sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Kruskal–Wallis: H = 1458.58
Kruskal–Wallis p-value: < 1e-300


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp

from scipy.stats import kruskal

# ----------------------------------------------------------
# Select EMT annotation column
# ----------------------------------------------------------
if (
    "cluster" in adata.obs.columns
    and set(["Epithelial", "Early Hybrid EMT", "Hybrid EMT", "Late EMT"])
    .issubset(set(adata.obs["cluster"].dropna()))
):
    group_col = "cluster"
    order = [
        "Epithelial",
        "Early Hybrid EMT",
        "Hybrid EMT",
        "Late EMT"
    ]

elif (
    "cluster_annotation" in adata.obs.columns
    and set(["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"])
    .issubset(set(adata.obs["cluster_annotation"].dropna()))
):
    group_col = "cluster_annotation"
    order = [
        "Epithelial",
        "Early-Hybrid",
        "Hybrid",
        "Late-EMT"
    ]

else:
    raise ValueError("Could not find EMT-state labels.")

score_col = "HPCS_ssGSEA"

# ----------------------------------------------------------
# Prepare data and statistics
# ----------------------------------------------------------
plot_df = (
    adata.obs[[group_col, score_col]]
    .dropna()
    .loc[lambda x: x[group_col].isin(order)]
    .copy()
)

counts = (
    plot_df.groupby(group_col, observed=True)
    .size()
    .reindex(order, fill_value=0)
)

medians = (
    plot_df.groupby(group_col, observed=True)[score_col]
    .median()
    .reindex(order)
)

groups = [
    plot_df.loc[plot_df[group_col] == group, score_col]
    for group in order
]

H, p_kw = kruskal(*groups)

dunn_p = sp.posthoc_dunn(
    plot_df,
    val_col=score_col,
    group_col=group_col,
    p_adjust="fdr_bh"
).reindex(index=order, columns=order)

display(dunn_p)

# ----------------------------------------------------------
# FIGURE 1: clean, zoomed HPCS violin plot
# ----------------------------------------------------------
sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(8.5, 5.5))

sns.violinplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    inner=None,
    cut=0,
    color="#5B89A6",
    linewidth=1.1,
    ax=ax
)

sns.boxplot(
    data=plot_df,
    x=group_col,
    y=score_col,
    order=order,
    width=0.10,
    showfliers=False,
    boxprops={
        "facecolor": "white",
        "edgecolor": "#222222",
        "linewidth": 1.1,
        "zorder": 4
    },
    whiskerprops={"color": "#222222", "linewidth": 1.1},
    capprops={"color": "#222222", "linewidth": 1.1},
    medianprops={"color": "#222222", "linewidth": 1.4},
    ax=ax
)

# Orange dashed median trend
ax.plot(
    range(len(order)),
    medians.values,
    color="#ff7f0e",
    linestyle="--",
    marker="o",
    markersize=5.5,
    linewidth=1.6,
    zorder=6,
    label="Median"
)

ax.set_xticks(
    range(len(order)),
    [f"{group}\n(n = {counts[group]:,})" for group in order],
    rotation=30,
    ha="right"
)

ax.set_xlabel("")
ax.set_ylabel("HPCS ssGSEA enrichment score")
ax.set_title("HPCS Program across EMT States")

# Tight y-axis: no statistical brackets taking up space
data_min = plot_df[score_col].min()
data_max = plot_df[score_col].max()
padding = (data_max - data_min) * 0.06

ax.set_ylim(data_min - padding, data_max + padding)

kw_text = "p < 1e-300" if p_kw == 0 else f"p = {p_kw:.2e}"

ax.text(
    0.02,
    0.98,
    f"Kruskal–Wallis: H = {H:.2f}, {kw_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9
)

ax.legend(frameon=False, loc="upper right")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# FIGURE 2: separate yellow Dunn-comparison heatmap
# ----------------------------------------------------------
# Retain lower triangle only
upper_triangle = np.triu(
    np.ones(dunn_p.shape, dtype=bool)
)

heatmap_p = dunn_p.mask(upper_triangle)

# Format p-values for display
labels = heatmap_p.copy().astype(object)

for row in heatmap_p.index:
    for col in heatmap_p.columns:
        p = heatmap_p.loc[row, col]

        if pd.isna(p):
            labels.loc[row, col] = ""
        elif p == 0:
            labels.loc[row, col] = "p < 1e-300"
        elif p < 0.001:
            labels.loc[row, col] = f"{p:.1e}"
        else:
            labels.loc[row, col] = f"{p:.3f}"

plt.figure(figsize=(7, 5.8))

sns.heatmap(
    heatmap_p.clip(upper=0.05),
    mask=heatmap_p.isna(),
    annot=labels,
    fmt="",
    cmap="viridis_r",   # yellow = low p-value
    vmin=0,
    vmax=0.05,
    square=True,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={"label": "Dunn test FDR-adjusted p-value"}
)

plt.title("HPCS Program: Pairwise Dunn Comparisons")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# If needed, run once in a separate Jupyter cell:
# %pip install gseapy scanpy

import gseapy as gp
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.sparse import issparse

# Set True ONLY if adata.X contains raw UMI counts.
# Keep False if adata.X is already normalized and log1p-transformed.
INPUT_IS_RAW_COUNTS = False
CHUNK_SIZE = 2000

# Mouse KS epithelial gene signature: 164 genes
KS_EPI_MOUSE = """
Cdh1 Agr2 Epcam Krt19 Rab25 Tacstd2 Galnt3 Fxyd3 Spint2 Tmem30b Scnn1a St14
Esrp1 S100a14 Cldn7 Erbb3 Rbm47 Spint1 Elf3 Cldn13 Prss8 Sh3yl1 Ehf Lcn2 Jup
Vamp8 Krt8 Inava Krt7 Dsp Cds1 Itgb4 Tmprss4 Lsr Sorl1 Grhl2 Ppl AA986860
Tspan1 Map7 Slpi Tox3 Arhgap8 F11r Lad1 Gpx2 Ctsh Adgrg1 Fa2h Klf5 Areg Krt18
Scel Cdh3 Ugt1a1 Mpzl2 Crybg1 Ovol2 Llgl2 Esrp2 Myo5c Ddr1 Vgll1 Irf6 Sfn
Tspan13 Kcnk1 Myo1d Pkp3 Itgb6 Ly75 Mapk13 Ttc39a Elmo3 Ceacam1 Dtx4 Erbb2
Rab11fip1 Atp2c2 Mst1r Ap1m2 Tgfa Myo6 Ptk6 Oas1g Fbp1 Aqp3 Cblc Epha1 Bspry
Eps8l1 Grb7 0610040J01Rik Klk6 Tjp2 Pls1 Dennd2d Eps8l2 Il20ra Hes1 Il1rn
Exph5 Arhgdib Misp Camk2n1 Hpgd Syngr2 Perp Mansc1 Dsc2 Pof1b Serinc5 Bik
Anxa9 Mall Epn3 Stap2 Foxa1 Pycard Slc37a1 Ank3 Tspan15 Hnmt Abcc3 Sdc1 Ckmt1
Tob1 B3gnt3 Tmc6 Cd9 Adap1 Atp1b1 Shank2 Cyb561 Ermp1 Rab20 Myh14 Capn1
Aldh3b2 Trim31 Arap2 Ssh3 Ica1 Arhgef5 Alox5 Rhod Tmprss2 Mtus1 Cyp4f18
Ppfibp2 Rabgap1l Plxnb2 Mgst2 Evpl Cd46 Krt15 Cnksr1 Blnk Comt Anxa4 Tnfsf13
Ocln Nherf1 Xbp1
""".split()

# Mouse KS mesenchymal gene signature: 48 genes
KS_MES_MOUSE = """
Vim Zeb1 Emp3 Sacs Axl Loxl2 Sparc Fhl4 Fermt2 Tuba1a Tmem158 Cald1 Lgals1
Pmp22 Msn Glyr1 Map1b Ap1s2 Gja1 Dennd5a Fam216a Tpm2 Tubb6 Srpx Ank2 Chn1
Sh2b3 P3h1 Etv1 Sobp Akap12 Tgfb1i1 Serpine1 Soat1 Col5a2 Lhfpl6 Cep170 Popdc3
Trpc1 Poglut2 Myl9 Bag2 Fstl1 Mxra7 Gfpt2 Reck Tmeff1 Cavin1
""".split()

# Optional preprocessing. This runs only if INPUT_IS_RAW_COUNTS = True.
if INPUT_IS_RAW_COUNTS:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

# Check whether the mouse signature genes match your adata.var_names
available_genes = set(adata.var_names.astype(str))
epi_found = [gene for gene in KS_EPI_MOUSE if gene in available_genes]
mes_found = [gene for gene in KS_MES_MOUSE if gene in available_genes]

print(f"KS-Epi genes found: {len(epi_found)}/{len(KS_EPI_MOUSE)}")
print(f"KS-Mes genes found: {len(mes_found)}/{len(KS_MES_MOUSE)}")

if len(epi_found) < 20 or len(mes_found) < 5:
    raise ValueError(
        "Too few signature genes matched. Your adata.var_names must contain "
        "mouse gene symbols, such as Cdh1, Epcam, and Vim—not Ensembl IDs."
    )

gene_sets = {
    "KS_Epi": KS_EPI_MOUSE,
    "KS_Mes": KS_MES_MOUSE,
}

all_scores = []

# ssGSEA in chunks, so it works with larger single-cell datasets
for start in range(0, adata.n_obs, CHUNK_SIZE):
    stop = min(start + CHUNK_SIZE, adata.n_obs)
    X = adata.X[start:stop]

    if issparse(X):
        X = X.T.toarray()
    else:
        X = np.asarray(X).T

    expression = pd.DataFrame(
        X,
        index=adata.var_names.astype(str),
        columns=adata.obs_names[start:stop].astype(str),
    )

    # Combine duplicate symbols, if present
    expression = expression.groupby(level=0, sort=False).mean()

    result = gp.ssgsea(
        data=expression,
        gene_sets=gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        min_size=5,
        max_size=500,
        outdir=None,
        no_plot=True,
        threads=1,
    )

    score_chunk = result.res2d.pivot(
        index="Name",
        columns="Term",
        values="ES",
    )

    score_chunk = score_chunk.reindex(
        adata.obs_names[start:stop].astype(str)
    )

    all_scores.append(score_chunk)
    print(f"Scored cells {start + 1:,}–{stop:,} of {adata.n_obs:,}")

scores = pd.concat(all_scores).reindex(adata.obs_names.astype(str))

# Write the ssGSEA scores directly into your loaded AnnData object
adata.obs["KS_Epi_ssGSEA"] = scores["KS_Epi"].to_numpy()
adata.obs["KS_Mes_ssGSEA"] = scores["KS_Mes"].to_numpy()

adata.uns["KS_ssGSEA_mouse"] = {
    "method": "ssGSEA with rank normalization",
    "epi_genes_found": len(epi_found),
    "epi_signature_size": len(KS_EPI_MOUSE),
    "mes_genes_found": len(mes_found),
    "mes_signature_size": len(KS_MES_MOUSE),
}

print("\nDone. Scores were added to adata.obs:")
display(adata.obs[["KS_Epi_ssGSEA", "KS_Mes_ssGSEA"]].head())

KS-Epi genes found: 158/164
KS-Mes genes found: 47/48
Scored cells 1–2,000 of 23,959
Scored cells 2,001–4,000 of 23,959
Scored cells 4,001–6,000 of 23,959
Scored cells 6,001–8,000 of 23,959
Scored cells 8,001–10,000 of 23,959
Scored cells 10,001–12,000 of 23,959
Scored cells 12,001–14,000 of 23,959
Scored cells 14,001–16,000 of 23,959
Scored cells 16,001–18,000 of 23,959
Scored cells 18,001–20,000 of 23,959
Scored cells 20,001–22,000 of 23,959
Scored cells 22,001–23,959 of 23,959

Done. Scores were added to adata.obs:


In [ ]:
output_file = "adata_KS_ssGSEA.h5ad"

adata.write_h5ad(
    output_file,
    compression="gzip"
)

print(f"Saved: {output_file}")

Saved: adata_KS_ssGSEA.h5ad


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# ----------------------------------------------------------
# Use the EXISTING EMT-state annotation from your dataset
# ----------------------------------------------------------
if (
    "cluster" in adata.obs.columns
    and set(["Epithelial", "Early Hybrid EMT", "Hybrid EMT", "Late EMT"])
    .issubset(set(adata.obs["cluster"].dropna()))
):
    group_col = "cluster"
    order = [
        "Epithelial",
        "Early Hybrid EMT",
        "Hybrid EMT",
        "Late EMT",
    ]

elif (
    "cluster_annotation" in adata.obs.columns
    and set(["Epithelial", "Early-Hybrid", "Hybrid", "Late-EMT"])
    .issubset(set(adata.obs["cluster_annotation"].dropna()))
):
    group_col = "cluster_annotation"
    order = [
        "Epithelial",
        "Early-Hybrid",
        "Hybrid",
        "Late-EMT",
    ]

else:
    raise ValueError(
        "Could not find the EMT labels in cluster or cluster_annotation."
    )

# ----------------------------------------------------------
# Prepare data: existing KS scores + existing EMT labels
# ----------------------------------------------------------
plot_df = (
    adata.obs[
        [group_col, "KS_Epi_ssGSEA", "KS_Mes_ssGSEA"]
    ]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .loc[lambda x: x[group_col].isin(order)]
    .copy()
)

# Pearson correlation across all cells
pearson_r, pearson_p = pearsonr(
    plot_df["KS_Epi_ssGSEA"],
    plot_df["KS_Mes_ssGSEA"],
)

print(f"Pearson correlation: r = {pearson_r:.3f}")
print(f"P-value: {pearson_p:.3e}")
print("\nCells per EMT state:")
print(plot_df[group_col].value_counts().reindex(order))

# ----------------------------------------------------------
# Scatter plot coloured by your existing EMT states
# ----------------------------------------------------------
palette = {
    order[0]: "#277DA1",  # Epithelial: blue
    order[1]: "#F4A261",  # Early Hybrid: orange
    order[2]: "#7B2CBF",  # Hybrid: purple
    order[3]: "#D62828",  # Late EMT: red
}

sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(8.5, 6.5))

sns.scatterplot(
    data=plot_df,
    x="KS_Epi_ssGSEA",
    y="KS_Mes_ssGSEA",
    hue=group_col,
    hue_order=order,
    palette=palette,
    s=12,
    alpha=0.45,
    linewidth=0,
    ax=ax,
)

# Overall linear trend
x = plot_df["KS_Epi_ssGSEA"].to_numpy()
y = plot_df["KS_Mes_ssGSEA"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)

x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(
    x_line,
    slope * x_line + intercept,
    color="black",
    linestyle="--",
    linewidth=1.3,
    label="Overall trend",
)

ax.set_xlabel("KS-Epi ssGSEA score", fontsize=12)
ax.set_ylabel("KS-Mes ssGSEA score", fontsize=12)
ax.set_title("KS-Epi versus KS-Mes across EMT States", fontsize=13)

p_text = "p < 1e-300" if pearson_p == 0 else f"p = {pearson_p:.2e}"

ax.text(
    0.03,
    0.97,
    f"Pearson r = {pearson_r:.3f}\n{p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
)

ax.legend(title="EMT state", frameon=False, loc="best")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Pearson correlation: r = -0.627
P-value: 0.000e+00

Cells per EMT state:
cluster
Epithelial           3980
Early Hybrid EMT    10175
Hybrid EMT           5715
Late EMT             4089
Name: count, dtype: int64
